## Who Worked On This Part?
Aarya Ray Chaudhuri: training, tuning and hosting the mushroom model on AWS SageMaker.

# Mushroom model on AWS SageMaker
This notebook trains and tunes our model on AWS and puts it online behind an HTTPS url, so the J backend can ask it for predictions.

It only runs inside **SageMaker Studio** (clone the repo there with the Git button and open this file from `Mushroom/`). It won't run on a laptop: it needs the AWS role of the SageMaker session. We keep the outputs in the repo, as the assignment asks for the SageMaker notebook.

The steps:
1. upload the raw csv to S3
2. one training job with the settings from notebook 2 (our baseline on AWS)
3. a hyperparameter tuning job that tries 12 settings
4. host the best model on an endpoint
5. put an HTTPS url with an api key in front of it (API Gateway + Lambda)

All the model code lives in `model/` and is the same code we test locally (`pytest tests` in `Mushroom/`). `MODEL_PLAN.md` explains every choice.

## Setup
SageMaker Studio now comes with version 3 of the SageMaker Python SDK, which works very differently. We pin version 2, the one the AWS course uses. After this cell, restart the kernel once (Kernel → Restart).

In [ ]:
%pip install -q "sagemaker>=2.257,<3"

We start a SageMaker session and get the role, the default S3 bucket and the region. In AWS Academy the role is the `LabRole`. The data path uses the same ROOT/DATA pattern as our other notebooks.

In [ ]:
import os
os.environ['SAGEMAKER_SUPPRESS_V2_WARNING'] = '1'

import json
import time
import zipfile
from pathlib import Path

import boto3
import sagemaker

# walk up from where the notebook runs to the repo root, so the data path works on every machine
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Mushroom' / 'datasets').is_dir())
MUSHROOM = ROOT / 'Mushroom'
DATA = MUSHROOM / 'datasets'

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
prefix = 'skooma-cats/mushroom'
region, bucket

## Which container
SageMaker runs our scripts in a ready-made scikit-learn container. We use the newest one, **1.9-0** (scikit-learn 1.9.0, pandas 2.2.3, Python 3.12). The model is trained **and** hosted in that same container, because a pickled sklearn model is only safe to load with the same sklearn version it was made with.

SDK version 2 doesn't know the name 1.9-0 yet, so we build the image address ourselves. The account numbers come from the SageMaker SDK's own list of images.

In [ ]:
# account that hosts the sagemaker scikit-learn images in each region
ACCOUNTS = {'us-east-1': '683313688378', 'us-east-2': '257758044811', 'us-west-2': '246618743249', 'eu-west-1': '141502667606', 'eu-central-1': '492215442770'}
# if the training job says the image doesn't exist, use '1.4-2-cpu-py3' (sklearn 1.4.2, our tests pass on it too)
IMAGE_TAG = '1.9-0-cpu-py3'
image = f'{ACCOUNTS[region]}.dkr.ecr.{region}.amazonaws.com/sagemaker-scikit-learn:{IMAGE_TAG}'
image

## Data to S3
We upload the **raw** csv, not `mushroom_clean.csv`. The clean file already has measurements filled in by an imputer that saw all 5000 rows, including the ones we test on. `train.py` does the cleaning steps itself and keeps the imputer inside the model, so it only learns from the training rows.

In [ ]:
train_s3 = session.upload_data(str(DATA / 'mushroom_project_dataset.csv'), bucket=bucket, key_prefix=f'{prefix}/data')
train_s3

## Baseline training job
First one job with the same random forest as in notebook 2 (200 trees, `min_samples_leaf=3`). `train.py` prints every metric as `name=value`. SageMaker reads them from the log with the regexes in `aws/metric_definitions.json`, and our tests check that those regexes find every metric.

In [ ]:
from sagemaker.sklearn import SKLearn

metric_definitions = json.loads((MUSHROOM / 'aws' / 'metric_definitions.json').read_text())
estimator = SKLearn(
    entry_point='train.py', source_dir=str(MUSHROOM / 'model'), image_uri=image, role=role,
    instance_type='ml.m5.large', instance_count=1,
    hyperparameters={'model': 'rf', 'n-estimators': 200, 'min-samples-leaf': 3},
    metric_definitions=metric_definitions, output_path=f's3://{bucket}/{prefix}/models',
    base_job_name='mushroom-rf', sagemaker_session=session,
)

This starts the job and waits for it. It takes a few minutes, most of it starting the machine.

In [ ]:
estimator.fit({'train': train_s3})

The final value of every metric, straight from the job description.

In [ ]:
job = session.sagemaker_client.describe_training_job(TrainingJobName=estimator.latest_training_job.name)
baseline = {m['MetricName']: m['Value'] for m in job['FinalMetricDataList']}
baseline, job['BillableTimeInSeconds']

_To write after the run: the baseline cv_auc and test numbers, and how they compare with the same model run locally (`python model/train.py`)._

## Hyperparameter tuning job
SageMaker trains 12 forests with different settings, 2 at a time, and uses what it learned from earlier jobs to pick the next settings (Bayesian search). What we tune:
- `n-estimators` (100 to 600): more trees average away more noise, but cost time
- `min-samples-leaf` (1 to 15): bigger leaves can't memorise single rows, which matters with flipped labels
- `max-features` (0.1 to 0.8): fewer features per split makes the trees more different from each other

The objective is **cv_auc**: the cross-validated AUC on the 80% training part. The 20% test part is never used to choose, otherwise we would be tuning on the test set (snoop bias).

In [ ]:
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

tuner = HyperparameterTuner(
    estimator, objective_metric_name='cv_auc', objective_type='Maximize', metric_definitions=metric_definitions,
    hyperparameter_ranges={
        'n-estimators': IntegerParameter(100, 600),
        'min-samples-leaf': IntegerParameter(1, 15),
        'max-features': ContinuousParameter(0.1, 0.8),
    },
    max_jobs=12, max_parallel_jobs=2, base_tuning_job_name='mushroom-rf-tune',
)

Start the tuning job and wait (around 20 to 30 minutes).

In [ ]:
tuner.fit({'train': train_s3})

Every try, best first. We save the table in `results/` so notebook 4 can compare it with the local models.

In [ ]:
trials = tuner.analytics().dataframe().sort_values('FinalObjectiveValue', ascending=False)
(MUSHROOM / 'results').mkdir(exist_ok=True)
trials.to_csv(MUSHROOM / 'results' / 'sagemaker_tuning.csv', index=False)
trials[['n-estimators', 'min-samples-leaf', 'max-features', 'FinalObjectiveValue', 'TrainingElapsedTimeSeconds']]

_To write after the run: which settings won, and whether the best cv_auc is really better than the baseline (a difference smaller than cv_auc_std is noise)._

## The best job
All metrics of the winning job, including the test numbers. We look at the test numbers only now, for the model we already picked. They are saved together with the place of the model file in S3.

In [ ]:
best_job = tuner.best_training_job()
best = session.sagemaker_client.describe_training_job(TrainingJobName=best_job)
best_result = {'job': best_job, 'hyperparameters': best['HyperParameters'],
               'metrics': {m['MetricName']: m['Value'] for m in best['FinalMetricDataList']},
               'model_data': best['ModelArtifacts']['S3ModelArtifacts']}
(MUSHROOM / 'results' / 'sagemaker_best.json').write_text(json.dumps(best_result, indent=2))
best_result['metrics']

## Host the model
The endpoint loads `model.tar.gz` and answers with `model/inference.py`. We host it on a normal endpoint on the smallest machine (`ml.t2.medium`). It is always warm, but it is paid per hour while it runs, so we turn it off when nobody needs it (last cell).

**Path we tried and dropped: serverless inference.** It only costs money when a request comes in, which would suit a website with a few requests a day. On 10 Oct the serverless endpoint crashed at startup. The CloudWatch log said `PermissionError: [Errno 13] Permission denied: '/etc/sagemaker-nginx.conf'`: the scikit-learn container writes its web server config to `/etc`, and a serverless endpoint runs the container on a read-only file system. That path is fixed inside AWS's container, so this container can't run serverless. Our own code never got to load. We keep the switch (`SERVERLESS`) to show what we tried.

The endpoint always gets the same name, so the url of the API never changes. This cell only needs the setup cells above it and `results/sagemaker_best.json`, so it is also the "turn it back on" cell.

In [ ]:
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.sklearn import SKLearnModel

ENDPOINT = 'skooma-mushroom'
# serverless crashes with this container (see above), so it stays off
SERVERLESS = False

best_result = json.loads((MUSHROOM / 'results' / 'sagemaker_best.json').read_text())
model = SKLearnModel(model_data=best_result['model_data'], role=role, entry_point='inference.py',
                     source_dir=str(MUSHROOM / 'model'), image_uri=image, sagemaker_session=session)
if SERVERLESS:
    predictor = model.deploy(endpoint_name=ENDPOINT, serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=2))
else:
    predictor = model.deploy(endpoint_name=ENDPOINT, initial_instance_count=1, instance_type='ml.t2.medium')

We send the four example requests from `aws/examples/` straight to the endpoint. `full`, `from_form` and `batch` should get a prediction, `invalid` should get an error message.

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name=region)
for name in ['full', 'from_form', 'batch', 'invalid']:
    body = (MUSHROOM / 'aws' / 'examples' / f'{name}.json').read_text()
    answer = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType='application/json', Body=body)
    print(name, answer['Body'].read().decode())

## A public HTTPS url for the backend
The endpoint itself only accepts requests signed with AWS keys, and AWS Academy keys expire after a few hours. So we put API Gateway in front of it: a public HTTPS url that checks an api key and passes the request to a small Lambda (`aws/lambda_function.py`), which calls the endpoint. Everything is described in `aws/gateway.yaml` (CloudFormation), so the whole thing can be rebuilt with this one cell.

First the Lambda code goes to S3, under a new name every time so CloudFormation notices a change.

In [ ]:
code_key = f'{prefix}/lambda/lambda-{int(time.time())}.zip'
with zipfile.ZipFile('/tmp/lambda.zip', 'w') as z:
    z.write(MUSHROOM / 'aws' / 'lambda_function.py', 'lambda_function.py')
boto3.client('s3', region_name=region).upload_file('/tmp/lambda.zip', bucket, code_key)
code_key

Then we create the stack, or update it if it already exists, and publish the routes to the `prod` stage. The Lambda runs with the same role as this notebook. If CloudFormation says Lambda can't use that role, put the LabRole arn here instead (IAM → Roles → LabRole).

In [ ]:
STACK = 'skooma-mushroom-api'
LAMBDA_ROLE = role
cfn = boto3.client('cloudformation', region_name=region)
params = [{'ParameterKey': k, 'ParameterValue': v} for k, v in
          {'EndpointName': ENDPOINT, 'LambdaRoleArn': LAMBDA_ROLE, 'CodeBucket': bucket, 'CodeKey': code_key}.items()]
template = (MUSHROOM / 'aws' / 'gateway.yaml').read_text()

existing = [s['StackName'] for s in cfn.list_stacks(StackStatusFilter=['CREATE_COMPLETE', 'UPDATE_COMPLETE', 'UPDATE_ROLLBACK_COMPLETE'])['StackSummaries']]
if STACK in existing:
    cfn.update_stack(StackName=STACK, TemplateBody=template, Parameters=params)
    cfn.get_waiter('stack_update_complete').wait(StackName=STACK)
else:
    cfn.create_stack(StackName=STACK, TemplateBody=template, Parameters=params)
    cfn.get_waiter('stack_create_complete').wait(StackName=STACK)

outputs = {o['OutputKey']: o['OutputValue'] for o in cfn.describe_stacks(StackName=STACK)['Stacks'][0]['Outputs']}
# cloudformation doesn't redeploy the api on an update, so we publish the current routes to the prod stage ourselves
boto3.client('apigateway', region_name=region).create_deployment(restApiId=outputs['ApiId'], stageName='prod')
outputs['PredictUrl']

The api key is a secret, so we don't print it (this notebook's output goes into git). We write it to a file in the Studio home folder, outside the repo, and share it with the backend team in private.

In [ ]:
api_key = boto3.client('apigateway', region_name=region).get_api_key(apiKey=outputs['ApiKeyId'], includeValue=True)['value']
(Path.home() / 'mushroom_api_key.txt').write_text(api_key)
print('api key saved, ends with', api_key[-4:])

## Calling it like the backend does
The same HTTPS request the J backend sends with curl: POST the mushroom as JSON, with the key in the `x-api-key` header.

In [ ]:
import urllib.request

body = (MUSHROOM / 'aws' / 'examples' / 'full.json').read_bytes()
request = urllib.request.Request(outputs['PredictUrl'], data=body, headers={'Content-Type': 'application/json', 'x-api-key': api_key})
with urllib.request.urlopen(request, timeout=30) as r:
    print(r.status, r.read().decode())

## Turning it off and on again
The api stack costs nothing while nobody calls it. A normal endpoint (`SERVERLESS = False`) costs money every hour, so delete it when we don't need it. The url and the key stay the same. To turn it back on: run the setup cells, then the "Host the model" cell.

In [ ]:
# set to True to delete the endpoint, the api stack stays so the url and key don't change
TURN_OFF = False
if TURN_OFF:
    predictor.delete_endpoint(delete_endpoint_config=True)

# Story
_To write after the run, with the real numbers: what the tuning job changed compared with the baseline job, whether that gain is bigger than the noise between folds, how long and how much it cost, and how the AWS model compares with the local models in notebook 4._